# 11 Attention 反向传播概述

前面我们已经学会了 Attention 的前向传播：输入怎样生成 $Q$、$K$、$V$，怎样得到注意力权重，最后怎样汇总 Value。

但模型只会前向计算还不够。训练时，它还要根据最终误差回答一个问题：

> 哪些参数应该改变，向哪个方向改变，才能让下一次预测更准确？

这一课先不急着逐项求导，只建立 Attention 反向传播的完整地图。

## 1. 为什么现在学习 Attention 的反向传播

我们之前已经学过通用反向传播、链式法则和 PyTorch 的 `loss.backward()`。

因此现在缺少的不是“反向传播是什么”，而是：

- Attention 内部有哪些计算节点；
- 梯度在这些节点之间怎样分流；
- $W_Q$、$W_K$、$W_V$ 分别从哪里收到学习信号；
- 同一个输入 $X$ 为什么会收到三条路径传回来的梯度。

这一课的重点是路线，不是背公式。

## 2. 先复习 Attention 的前向传播

单头 Self-Attention 的核心计算是：

$$
\begin{aligned}
Q &= XW_Q, \\
K &= XW_K, \\
V &= XW_V, \\
S &= \frac{QK^{\top}}{\sqrt{d_k}}, \\
A &= \operatorname{softmax}(S), \\
O &= AV.
\end{aligned}
$$

前向传播沿着下面的方向计算：

$$
X\rightarrow(Q,K,V)\rightarrow S\rightarrow A\rightarrow O\rightarrow\mathcal{L}
$$

其中 $\mathcal{L}$ 表示最终的损失。

## 3. 反向传播就是沿计算图倒着追责

前向传播得到损失，反向传播从损失出发，沿原路线反向传递梯度：

$$
\mathcal{L}\rightarrow O\rightarrow(A,V)\rightarrow S\rightarrow(Q,K)\rightarrow(W_Q,W_K,W_V,X)
$$

这里的“追责”不是判断某个 token 对不对，而是在计算：

$$
\frac{\partial\mathcal{L}}{\partial Z}
$$

它表示：如果中间量 $Z$ 发生一点变化，最终损失会怎样变化。

## 4. 第一处分流：$O=AV$

$O$ 同时由注意力权重 $A$ 和 Value $V$ 决定。

所以损失传到 $O$ 后，会分成两条路线：

$$
\frac{\partial\mathcal{L}}{\partial O}\rightarrow\left\{\begin{array}{l}\dfrac{\partial\mathcal{L}}{\partial A}:\text{注意力分配是否需要调整}\\[6pt]\dfrac{\partial\mathcal{L}}{\partial V}:\text{被汇总的内容是否需要调整}\end{array}\right.
$$

直觉上：

- $A$ 决定“从谁那里取多少信息”；
- $V$ 决定“真正取到什么信息”。

预测错误时，这两部分都有可能需要改变。

## 5. 权重路线：$A\rightarrow S\rightarrow Q,K$

$A$ 不是模型直接保存的一张固定表，它由分数 $S$ 经过 Softmax 动态产生。

因此注意力权重路线继续向前追：

$$
\frac{\partial\mathcal{L}}{\partial A}\rightarrow\frac{\partial\mathcal{L}}{\partial S}\rightarrow\left(\frac{\partial\mathcal{L}}{\partial Q},\frac{\partial\mathcal{L}}{\partial K}\right)
$$

它表达的是：

1. 哪些注意力权重导致了当前误差；
2. 要改变这些权重，哪些匹配分数应该提高或降低；
3. 要改变匹配分数，Query 和 Key 应该怎样改变。

## 6. 三组参数怎样收到梯度

因为 $Q$、$K$、$V$ 都由线性变换得到，所以梯度会继续传给三组参数：

$$
\begin{aligned}
Q=XW_Q&\quad\Longrightarrow\quad\frac{\partial\mathcal{L}}{\partial W_Q},\\
K=XW_K&\quad\Longrightarrow\quad\frac{\partial\mathcal{L}}{\partial W_K},\\
V=XW_V&\quad\Longrightarrow\quad\frac{\partial\mathcal{L}}{\partial W_V}.
\end{aligned}
$$

于是三组参数学习的侧重点不同：

- $W_Q$ 学习怎样提出更合适的查询；
- $W_K$ 学习怎样提供更合适的匹配依据；
- $W_V$ 学习怎样提供更有用的内容。

所谓 QKV 的“意义”，不是人工写死的，而是在大量训练样本中通过这些梯度逐渐形成的。

## 7. 输入 $X$ 会收到三条路线的梯度

同一个 $X$ 同时参与生成 $Q$、$K$ 和 $V$：

$$
X\rightarrow\left\{\begin{array}{l}Q=XW_Q\\K=XW_K\\V=XW_V\end{array}\right.
$$

因此反向传播回来时，$X$ 的总梯度不是只来自一条路线，而是三条路线相加：

$$
\frac{\partial\mathcal{L}}{\partial X}=\left.\frac{\partial\mathcal{L}}{\partial X}\right|_Q+\left.\frac{\partial\mathcal{L}}{\partial X}\right|_K+\left.\frac{\partial\mathcal{L}}{\partial X}\right|_V
$$

这是反向传播的一条通用规则：一个变量如果在前向传播中走向多个分支，反向传播时各分支的梯度要汇总。

## 8. 梯度的形状和对应变量相同

假设不考虑 batch：

$$
\begin{aligned}
X&:N\times D, & Q,K&:N\times d_k,\\
V,O&:N\times d_v, & A,S&:N\times N.
\end{aligned}
$$

那么对应梯度的形状分别是：

$$
\begin{aligned}
\frac{\partial\mathcal{L}}{\partial X}&:N\times D, & \frac{\partial\mathcal{L}}{\partial Q},\frac{\partial\mathcal{L}}{\partial K}&:N\times d_k,\\
\frac{\partial\mathcal{L}}{\partial V},\frac{\partial\mathcal{L}}{\partial O}&:N\times d_v, & \frac{\partial\mathcal{L}}{\partial A},\frac{\partial\mathcal{L}}{\partial S}&:N\times N.
\end{aligned}
$$

检查梯度形状，是以后排查矩阵推导和代码错误的重要方法。

## 9. $\sqrt{d_k}$ 在反向传播中做什么

前向传播中：

$$
S=\frac{QK^{\top}}{\sqrt{d_k}}
$$

$d_k$ 是人为设计的维度，不是训练参数，因此不会学习，也不需要梯度。

它只是一个固定缩放因子。反向传播经过这里时，传给 $QK^{\top}$ 的梯度同样会乘以 $1/\sqrt{d_k}$。

所以它既控制前向分数的尺度，也控制这一段反向梯度的尺度。

## 10. 放回 Transformer Encoder 中看

真实的 Transformer Encoder 不只有 Attention，还包含残差连接、LayerNorm 和 FFN。

因此完整梯度路线会更长：

$$
\mathcal{L}\rightarrow\operatorname{FFN}\rightarrow\operatorname{LayerNorm}\rightarrow\text{残差分支}\rightarrow\operatorname{Attention}
$$

这一阶段我们先把 Attention 子模块单独拿出来看。

这和以前单独学习 MLP、卷积层的反向传播一样：先理解局部模块，再理解整个网络。

## 11. 三个常见误解

### 误解一：注意力权重 $A$ 是被优化器直接更新的参数

不是。$A$ 是当前输入经过计算临时得到的中间结果。优化器真正更新的是 $W_Q$、$W_K$、$W_V$ 等参数。

### 误解二：只有 $V$ 会收到梯度

不是。因为输出同时依赖 $A$ 和 $V$，所以权重路线和内容路线都会收到梯度。

### 误解三：调用 `backward()` 就等于已经更新参数

不是。`backward()` 负责计算并保存梯度；优化器的 `step()` 才根据梯度更新参数。

## 12. 本节小结

1. Attention 的反向传播从损失出发，沿前向计算图倒序传递。
2. 在 $O=AV$ 处，梯度分给注意力权重 $A$ 和内容 $V$。
3. $A$ 的梯度继续经过 Softmax 和分数矩阵，传给 $Q$、$K$。
4. $Q$、$K$、$V$ 的梯度继续传给 $W_Q$、$W_K$、$W_V$。
5. 输入 $X$ 同时参与三条路线，因此它的总梯度是三条路线之和。
6. 梯度张量的形状与它所对应的变量形状相同。
7. PyTorch 能自动完成计算，但理解路线可以帮助我们解释模型和排查错误。

## 13. 自测问题

1. Attention 的前向传播和反向传播方向有什么区别？
2. 为什么在 $O=AV$ 处，梯度会分成两条路线？
3. 注意力权重 $A$ 是模型参数吗？优化器会直接更新它吗？
4. $W_Q$、$W_K$、$W_V$ 分别通过哪条路线收到梯度？
5. 为什么输入 $X$ 的梯度要由三部分相加？
6. 如果 $A$ 的形状是 $N\times N$，它的梯度是什么形状？
7. `loss.backward()` 和 `optimizer.step()` 的职责有什么区别？